# Reinforcement Learning

# 2. Online control

This notebook presents the **online control** of an agent by Q-learning and policy gradient.

In [19]:
import numpy as np

In [20]:
from model import TicTacToe, Nim, ConnectFour
from agent import Agent, OnlineControl
from dynamic import ValueIteration

## To do

* Complete the class ``QLearning``.
* Test it on Tic-Tac-Toe.
* Test the algorithm on Connect 4 against a random adversary. Comment the results.

## Q-learning

In [21]:
class QLearning(OnlineControl):
    """Online control by Q-learning."""

    def train(self, horizon=100, n_episodes=100, epsilon=0.1):
            """Learn the action-value function online.
            
            Parameters
            ----------
            horizon : int
                Time horizon of the episode.
            n_episodes : int
                Number of episodes.
            epsilon : float
                Exploration rate.
            """
            for episode in range(n_episodes):
                self.environment.reset()
                state = self.environment.state
                for t in range(horizon):
                    # encode the state to get a valid key
                    code = self.environment.encode(state)
                    # get action
                    action = self.randomize_best_action(state, epsilon=epsilon)
                    reward, stop = self.environment.step(action)
                    next_state = self.environment.state
                    # sample of the return for the current state-action
                    if stop:
                        sample = reward
                    else:
                        next_code = self.environment.encode(next_state)
                        actions = self.get_actions(next_state)
                        next_values = [self.action_value[next_code][a] for a in actions]
                        if self.player == 1:
                            sample = reward + self.gamma * np.max(next_values)
                        else:
                            sample = reward + self.gamma * np.min(next_values)
                    # update the Q function
                    self.action_count[code][action] += 1
                    diff = sample - self.action_value[code][action]
                    count = self.action_count[code][action]
                    self.action_value[code][action] += diff / count
                    if stop:
                        break
                    state = next_state             

## Test

In [22]:
def mean_return(agent, n_episodes=100, horizon=100):
    return float(np.mean(agent.get_returns(n_episodes=n_episodes, horizon=horizon)))

In [23]:
np.random.seed(0)

game = TicTacToe()
agent = QLearning(game)
print("TicTacToe before training:", mean_return(agent, n_episodes=300, horizon=9))

TicTacToe before training: 0.34


In [24]:
agent.train(horizon=9, n_episodes=1000, epsilon=0.1)
agent.update_policy()
print("TicTacToe after training:", mean_return(agent, n_episodes=300, horizon=9))

TicTacToe after training: 0.7766666666666666


In [25]:
np.random.seed(0)

game = ConnectFour()
agent = QLearning(game)
print("ConnectFour before training:", mean_return(agent, n_episodes=100, horizon=50))
agent.train(horizon=50, n_episodes=100, epsilon=0.1)
agent.update_policy()
print("ConnectFour after training:", mean_return(agent, n_episodes=100, horizon=50))

ConnectFour before training: 0.18
ConnectFour after training: 0.04


## Comments

With the fixed seed above, Q-learning improves the mean return on **TicTacToe** from about **0.34** to about **0.78**. This is expected because the state space is small enough for a tabular method to learn a useful Q-function.

On **ConnectFour**, the result is much poorer: the mean return stays low and can even decrease slightly after training. The state space is far too large for tabular Q-learning with so few episodes, so the learned Q-values remain sparse and unstable.

## Policy gradient

We now consider policy gradient, applicable to a large state space. A neural network is used to approximate the optimal policy. It returns the probability of each action. We use a neural network with a single hidden layer.

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.set_num_threads(1)

In [27]:
class Classifier(nn.Module):
    """Neural network for policy gradient. Return the distribution of actions in each state.
    
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    layers : list of int
        Dimensions of the hidden layers.
    """
    def __init__(self, environment, layers):
        if not hasattr(environment, 'one_hot_encode'):
            raise ValueError("The environment must have a one-hot encoding of states.")   
        super(Classifier, self).__init__()
        self.environment = environment
        actions = environment.get_all_actions()
        if self.environment.is_game():
            # remove pass action
            actions = [action for action in actions if action != "pass"]
        self.actions = actions
        state = environment.init_state()
        code = environment.one_hot_encode(state)
        input_size = len(code)
        output_size = len(actions)
        self.nn = nn.Sequential()
        for number, size in enumerate(layers):
            output_size = size
            self.nn.add_module(f"Linear {number}", nn.Linear(input_size, output_size))
            self.nn.add_module(f"Activation {number}", nn.ReLU())
            input_size = size
        self.nn.add_module("Output", nn.Linear(input_size, len(actions)))
        self.nn.add_module("Softmax", nn.Softmax(dim=0))

    def forward(self, code):
        """Forward pass."""
        return self.nn(code)

## Test

In [29]:
game = TicTacToe()

In [30]:
classifier = Classifier(environment=game, layers=[10, 5])

In [31]:
classifier.nn

Sequential(
  (Linear 0): Linear(in_features=18, out_features=10, bias=True)
  (Activation 0): ReLU()
  (Linear 1): Linear(in_features=10, out_features=5, bias=True)
  (Activation 1): ReLU()
  (Output): Linear(in_features=5, out_features=9, bias=True)
  (Softmax): Softmax(dim=0)
)

In [32]:
state = game.state
# one-hot encoding
code = game.one_hot_encode(state)
# tensor
code = torch.tensor(code).float()

In [33]:
# probability of each action before training
probs = classifier.forward(code).detach()
print(probs)

tensor([0.1019, 0.1174, 0.0965, 0.0846, 0.1344, 0.0859, 0.1463, 0.1430, 0.0901])


In [34]:
classifier.actions

[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]

In [35]:
torch.sum(probs)

tensor(1.0000)

## To do

* Complete the method `train` of the class PolicyGradient. Observe that a penalty is assigned for illegal actions.
* Test the agent on TicTacToe, against (1) a random adversary and (2) a perfect adversary.
* Test the agent on ConnectFour, against (1) a random adversary and (2) an adversary with the one-step policy.
* Compare our results to Q-learning.

In [36]:
class PolicyGradient(Agent):
    """Agent learning by policy gradient.
    
    Parameters
    ----------
    environment : object of class Environment
        Environment of the agent.
    player : int
        Player for games (1 or -1, default = default player of the game).
    gamma : float
        Discount rate (in [0, 1], default = 1).
    layers : list of int
        Dimensions of the hidden layers.
    penalty : float
        Penalty for illegal actions (default = -5).
    min_log : float
        Minimal value to compute the log (default = 1e-10)
    """
    def __init__(self, environment, player=None, gamma=1, layers=[10, 5], penalty=-5, min_log=1e-10):
        super(PolicyGradient, self).__init__(environment=environment, player=player)  
        self.nn = Classifier(environment, layers)
        self.action_id = {action: i for i, action in enumerate(self.nn.actions)}
        self.gamma = gamma
        self.penalty = penalty
        self.min_log = min_log
        
    def get_policy(self):
        """Get the current policy."""
        def policy(state):
            actions = self.environment.get_actions(state)
            if len(actions) > 1:
                win_actions = []
                # check win actions
                if self.environment.is_game():
                    next_states = [self.environment.get_next_state(state, action) for action in actions]
                    win_actions = [self.environment.get_reward(next_state) == self.player for next_state in next_states]
                if any(win_actions):
                    probs = np.array(win_actions).astype(float)
                else:
                    # get prob of each action
                    code = self.environment.one_hot_encode(state)
                    code = torch.tensor(code).float()
                    probs = self.nn.forward(code)
                    probs = probs.detach().numpy()
                    # restrict to available actions
                    indices = [self.action_id[action] for action in actions]
                    probs = probs[indices]                    
                # renormalize
                if np.sum(probs) > 0:
                    probs = probs / np.sum(probs)
                else:
                    probs = np.ones(len(actions)) / len(actions)
            else:
                probs = [1]
            return probs, actions
        return policy
    
    def update_policy(self):
        """Update the policy (after training)."""
        self.policy = self.get_policy()
    
    def get_samples(self, horizon):
        """Get samples from one episode (return and log-probs of the actions)."""
        rewards = []
        log_probs = []
        log_probs_illegal = []
        self.environment.reset()
        state = self.environment.state
        for t in range(horizon):
            actions = self.environment.get_actions(state)
            if actions == ["pass"]:
                reward, stop = self.environment.step("pass")
                rewards.append(reward)
                state = self.environment.state
            else:
                code = self.environment.one_hot_encode(state)
                code = torch.tensor(code).float()
                probs = self.nn.forward(code)
                i = np.random.choice(len(self.nn.actions), p=probs.detach().numpy())
                action = self.nn.actions[i]
                prob = torch.clip(probs[i], self.min_log, 1 - self.min_log)
                log_prob = torch.log(prob).reshape(1)
                if action in actions:
                    reward, stop = self.environment.step(action)
                    state = self.environment.state
                    rewards.append(reward)
                    log_probs.append(log_prob)
                else:
                    stop = False
                    log_probs_illegal.append(log_prob)
            if stop:
                break
        return_ = 0
        for reward in reversed(rewards):
            return_ = reward + self.gamma * return_
        return return_, log_probs, log_probs_illegal
        
    def train(self, horizon=100, n_episodes=1000, batch_size=10, learning_rate=0.01):
        """Train the neural network.
                    
            Parameters
            ----------
            horizon : int
                Time horizon of the episode.
            n_episodes : int
                Number of episodes.
            batch_size : int
                Batch size (in number of episodes).
            learning_rate : float
                Learning rate.
        """
        optimizer = optim.Adam(self.nn.parameters(), lr=learning_rate)
        loss = None
        count = 0
        average = 0
        for episode in range(n_episodes):
            return_, log_probs, log_probs_illegal = self.get_samples(horizon)
            advantage = return_ - average
            count += 1
            average += (return_ - average) / count
            episode_loss = None
            if len(log_probs):
                term = - advantage * torch.sum(torch.cat(log_probs))
                episode_loss = term if episode_loss is None else episode_loss + term
            if len(log_probs_illegal):
                term = - self.penalty * torch.sum(torch.cat(log_probs_illegal))
                episode_loss = term if episode_loss is None else episode_loss + term
            if episode_loss is not None:
                loss = episode_loss if loss is None else loss + episode_loss
            if ((episode + 1) % batch_size == 0 or episode == n_episodes - 1) and loss is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                loss = None

In [37]:
np.random.seed(0)
torch.manual_seed(0)

game = TicTacToe()
agent = PolicyGradient(game)
print("TicTacToe vs random before training:", mean_return(agent, n_episodes=300, horizon=9))

TicTacToe vs random before training: 0.34


In [38]:
agent.train(horizon=9, n_episodes=200, batch_size=20, learning_rate=0.01)
agent.update_policy()
print("TicTacToe vs random after training:", mean_return(agent, n_episodes=300, horizon=9))

TicTacToe vs random after training: 0.67


In [39]:
solver = ValueIteration(TicTacToe(), gamma=1, n_iter=9)
_, perfect_adversary = solver.get_perfect_players()

np.random.seed(0)
torch.manual_seed(0)
game = TicTacToe(adversary_policy=perfect_adversary)
agent = PolicyGradient(game)
print("TicTacToe vs perfect adversary before training:", mean_return(agent, n_episodes=300, horizon=9))
agent.train(horizon=9, n_episodes=200, batch_size=20, learning_rate=0.01)
agent.update_policy()
print("TicTacToe vs perfect adversary after training:", mean_return(agent, n_episodes=300, horizon=9))

TicTacToe vs perfect adversary before training: -0.7766666666666666
TicTacToe vs perfect adversary after training: -0.7433333333333333


In [40]:
np.random.seed(0)
torch.manual_seed(0)

game = ConnectFour()
agent = PolicyGradient(game, layers=[16, 8])
print("ConnectFour vs random before training:", mean_return(agent, n_episodes=100, horizon=50))
agent.train(horizon=50, n_episodes=20, batch_size=10, learning_rate=0.005)
agent.update_policy()
print("ConnectFour vs random after training:", mean_return(agent, n_episodes=100, horizon=50))

ConnectFour vs random before training: 0.18
ConnectFour vs random after training: 0.72


In [41]:
np.random.seed(0)
torch.manual_seed(0)

game = ConnectFour(adversary_policy='one_step')
agent = PolicyGradient(game, layers=[16, 8])
print("ConnectFour vs one-step before training:", mean_return(agent, n_episodes=100, horizon=50))
agent.train(horizon=50, n_episodes=20, batch_size=10, learning_rate=0.005)
agent.update_policy()
print("ConnectFour vs one-step after training:", mean_return(agent, n_episodes=100, horizon=50))

ConnectFour vs one-step before training: -0.94
ConnectFour vs one-step after training: -0.62


## Answers

With the fixed seeds used above, **PolicyGradient** improves on **TicTacToe** against a random adversary: the mean return goes from about **0.34** to about **0.67**. Against a **perfect adversary**, the return stays negative, around **-0.78** before training and **-0.74** after training, so the learned policy is still losing.

On **ConnectFour**, the policy-gradient approach behaves much better than tabular Q-learning. Against a **random adversary**, the mean return increases from about **0.18** to about **0.72**. Against a **one-step adversary**, it also improves, from about **-0.94** to about **-0.62**, but it remains negative.

Compared with **Q-learning**, the conclusion is clear: **Q-learning is better on the small state space of TicTacToe**, while **PolicyGradient is more suitable for the larger state space of ConnectFour**.